In [3]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
from rdkit.Chem import MACCSkeys

# Input
df = pd.read_csv("smiles_all_cids.csv", sep=",")

In [6]:


# Adjust column names if necessary
results = []

# SMARTS patterns
patterns = {
    "carboxylic_acid": "[CX3](=O)[OX2H1]",
    "phenol": "[c][OX2H1]",
    "aromatic_ring": "a1aaaaa1",
    "indole": "c1ccc2[nH]ccc2c1",
    "amine": "[NX3;H2,H1;!$(NC=O)]",
    "amide": "[NX3][CX3](=[OX1])",
    "sulfate": "[SX4](=O)(=O)([O-])O",
    "phosphate": "[PX4](=O)(O)(O)O",
    "ester": "[CX3](=O)[OX2][#6]",
}

compiled_patterns = {
    name: Chem.MolFromSmarts(smarts)
    for name, smarts in patterns.items()
}

for _, row in df.iterrows():

    cid = row["CID"]
    smiles = row["SMILES"]

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        results.append({
            "CID": cid,
            "SMILES": smiles,
            "valid": False
        })
        continue

    record = {
        "CID": cid,
        "SMILES": smiles,
        "valid": True,

        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),

        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),

        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "Carbons": sum(
            atom.GetAtomicNum() == 6
            for atom in mol.GetAtoms()
        ),
    }

    for name, pattern in compiled_patterns.items():
        record[name] = mol.HasSubstructMatch(pattern)

    results.append(record)


[16:23:11] WARNING: not removing hydrogen atom without neighbors


In [7]:
results

[{'CID': '33032',
  'SMILES': 'C(CC(=O)O)[C@@H](C(=O)O)N',
  'valid': True,
  'MW': 147.13,
  'LogP': -0.736899999999999,
  'TPSA': 100.61999999999999,
  'HBD': 3,
  'HBA': 3,
  'RotatableBonds': 4,
  'Rings': 0,
  'AromaticRings': 0,
  'Carbons': 5,
  'carboxylic_acid': True,
  'phenol': False,
  'aromatic_ring': False,
  'indole': False,
  'amine': True,
  'amide': False,
  'sulfate': False,
  'phosphate': False,
  'ester': False},
 {'CID': '6288',
  'SMILES': 'C[C@H]([C@@H](C(=O)O)N)O',
  'valid': True,
  'MW': 119.11999999999999,
  'LogP': -1.2208999999999997,
  'TPSA': 83.55,
  'HBD': 3,
  'HBA': 3,
  'RotatableBonds': 2,
  'Rings': 0,
  'AromaticRings': 0,
  'Carbons': 4,
  'carboxylic_acid': True,
  'phenol': False,
  'aromatic_ring': False,
  'indole': False,
  'amine': True,
  'amide': False,
  'sulfate': False,
  'phosphate': False,
  'ester': False},
 {'CID': '104745',
  'SMILES': 'CCC(=O)[O-]',
  'valid': True,
  'MW': 73.071,
  'LogP': -0.8537000000000001,
  'TPSA': 40.129

In [8]:
# Save
out = pd.DataFrame(results)

In [13]:
out.iloc[1,:]

CID                                    6288
SMILES             C[C@H]([C@@H](C(=O)O)N)O
valid                                  True
MW                                   119.12
LogP                                -1.2209
TPSA                                  83.55
HBD                                       3
HBA                                       3
RotatableBonds                            2
Rings                                     0
AromaticRings                             0
Carbons                                   4
carboxylic_acid                        True
phenol                                False
aromatic_ring                         False
indole                                False
amine                                  True
amide                                 False
sulfate                               False
phosphate                             False
ester                                 False
Name: 1, dtype: object

In [ ]:


out.to_csv("rdkit_descriptors.tsv", sep="\t", index=False)

print(out.shape)
print(out.head())
